# =========================================================================
# Project:
- Customer Relationship Management

# Module:
- JoDongHwi

# Notebook:
- libght_gbm.ipynb

# Purpose:
- dataset을 이용해서 LightGBM으로 모델 학습
  목표는 아래와 같음
  
  - Accuracy: ≥ 0.80
  - Recall: ≥ 0.70
  - precision: ≥ 0.70
  - F1 Score: ≥ 0.78
  - ROC-AUC: ≥ 0.85
  - PR-AUC: ≥ 0.80
  

# Author:
- @nobrain711

# Created:
- 2026-03-11

# Updated:
- 2026-03-11: initial version (@nobrain711)
- 2026-03-13: common mlflow 모듈 적용 (@nobrain711)
# =========================================================================


# 라이브러리 import

In [1]:
# data
import numpy as np
import pandas as pd

# model
import lightgbm as lgb
from lightgbm import LGBMClassifier

# split
from sklearn.model_selection import train_test_split

# metrics
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
)

# visualzation
import matplotlib.pyplot as plt

# dataset import
from common.fetch_creditcard import fetch_creditcard

# mlflow module
from common.mlflow import (
    end_run,
    log_artifact,
    log_model,
    log_metrics,
    log_params,
    log_tags,
    reset_active_run,
    setup_mlflow,
    start_run,
)

# Dataset Load

In [2]:
# fetch_creditcard로 data 로드
X, y = fetch_creditcard(X_y_split=True)

print("=" * 40)
print("Dataset Shape Summary")
print("=" * 40)
print(f"X (Features) : {X.shape}")
print(f"y (Target)   : {y.shape}")
print("-" * 40)
print("Target Distribution")
print(y.value_counts())
print("=" * 40)

Dataset Shape Summary
X (Features) : (10227, 20)
y (Target)   : (10227,)
----------------------------------------
Target Distribution
0    8551
1    1676
Name: count, dtype: int64


In [3]:
# X 데이터 확인
display(X.info())

<class 'pandas.core.frame.DataFrame'>
Index: 10227 entries, 1 to 10254
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   creditcard_churn_id  10227 non-null  int64  
 1   age                  10227 non-null  int64  
 2   gender               10227 non-null  int64  
 3   dependents           10227 non-null  int64  
 4   education_id         10227 non-null  int64  
 5   marital_id           10227 non-null  int64  
 6   income_id            10227 non-null  int64  
 7   card_type_id         10227 non-null  int64  
 8   relationship_months  10227 non-null  int64  
 9   product_count        10227 non-null  int64  
 10  inactive_months      10227 non-null  int64  
 11  contact_count        10227 non-null  int64  
 12  credit_limit         10227 non-null  float64
 13  revolving_balance    10227 non-null  float64
 14  available_credit     10227 non-null  float64
 15  amount_change        10227 non-null  floa

None

## Trainset, Testset Split

In [4]:
# feature / target 준비
categorical_features = [
    "gender",
    "education_id",
    "marital_id",
    "income_id",
    "card_type_id",
]

X_model = X.drop(columns=["creditcard_churn_id"]).copy()
y_model = y.copy()

In [5]:
# pandas category 타입으로 변환
for col in categorical_features:
    X_model[col] = X_model[col].astype("category")

In [6]:
print("Training Dataset Summary")
print("=" * 40)
print(f"X_model shape : {X_model.shape}")
print(f"y_model shape : {y_model.shape}")
print("-" * 40)
print("Categorical Features")
for col in categorical_features:
    print(f"- {col}: {X_model[col].dtype}")
print("=" * 40)

Training Dataset Summary
X_model shape : (10227, 19)
y_model shape : (10227,)
----------------------------------------
Categorical Features
- gender: category
- education_id: category
- marital_id: category
- income_id: category
- card_type_id: category


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X_model, y_model, test_size=0.2, random_state=42, stratify=y
)

print(f"{'Train / Test Split Summary':^50}")
print("=" * 50)

print(f"X_train shape : {X_train.shape}")
print(f"X_test  shape : {X_test.shape}")
print("-" * 50)
print(f"y_train shape : {y_train.shape}")
print(f"y_test  shape : {y_test.shape}")

print("-" * 50)
print("Target Distribution (Train)")
print(y_train.value_counts())

print("-" * 50)
print("Target Distribution (Test)")
print(y_test.value_counts())

print("-" * 50)
print("Train Ratio")
print(y_train.value_counts(normalize=True))

print("-" * 50)
print("Test Ratio")
print(y_test.value_counts(normalize=True))

print("-" * 50)
print("Class Imbalance Ratio")
print(f"Churn Rate : {y_train.mean():.3f}")

print("=" * 50)

            Train / Test Split Summary            
X_train shape : (8181, 19)
X_test  shape : (2046, 19)
--------------------------------------------------
y_train shape : (8181,)
y_test  shape : (2046,)
--------------------------------------------------
Target Distribution (Train)
0    6840
1    1341
Name: count, dtype: int64
--------------------------------------------------
Target Distribution (Test)
0    1711
1     335
Name: count, dtype: int64
--------------------------------------------------
Train Ratio
0    0.836084
1    0.163916
Name: proportion, dtype: float64
--------------------------------------------------
Test Ratio
0    0.836266
1    0.163734
Name: proportion, dtype: float64
--------------------------------------------------
Class Imbalance Ratio
Churn Rate : 0.164


# Model (LightGBM baseline) Train

In [20]:
# mlflow에 훈련 등록
reset_active_run()
start_run(
    run_name="lightgbm_baseline",
    tags={
        "project": "ccrm",
        "model_name": "LightGBM",
        "task": "classification",
        "stage": "baseline",
    },
)


<ActiveRun: >

In [21]:
# model params
params = {
    "objective": "binary",
    "n_estimators": 200,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": 6,
    "random_state": 42,
    "n_jobs": -1,
    "is_unbalance": True
}

In [22]:
# model params setting
model = LGBMClassifier(**params)

log_params(params)

In [23]:
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1_score": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_proba),
    "pr_auc": average_precision_score(y_test, y_proba),
}

[LightGBM] [Info] Number of positive: 1341, number of negative: 6840
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000525 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2056
[LightGBM] [Info] Number of data points in the train set: 8181, number of used features: 19
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.163916 -> initscore=-1.629372
[LightGBM] [Info] Start training from score -1.629372
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: 

In [24]:
log_model(model, "lightgbm")

log_metrics(metrics)

print("=== metrics ===")
print(pd.Series(metrics))
print()
print(classification_report(y_test, y_pred))

2026/03/13 00:59:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/13 00:59:53 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


=== metrics ===
accuracy     0.962854
precision    0.850949
recall       0.937313
f1_score     0.892045
roc_auc      0.989780
pr_auc       0.968125
dtype: float64

              precision    recall  f1-score   support

           0       0.99      0.97      0.98      1711
           1       0.85      0.94      0.89       335

    accuracy                           0.96      2046
   macro avg       0.92      0.95      0.93      2046
weighted avg       0.97      0.96      0.96      2046



In [25]:
end_run()

🏃 View run lightgbm_baseline at: http://mlflow:5000/#/experiments/1/runs/4d7e8e8ab409485f97052a79b13dc7f5
🧪 View experiment at: http://mlflow:5000/#/experiments/1
